# CNN Model Development - NIH Chest X-ray Classification

### Objective

This notebook develops a simple Convolutional Neural Network (CNN) using PyTorch to classify chest X-ray images into two categories: Effusion and No Finding.

### Execution Plan

In this notebook, we will
1. Load existing dataset and construct PyTorch DataLoaders
2. Build CNN architecture
3. Inspect model parameters and tensor dimensions
4. Train CNN using native PyTorch training loop
5. Evaluate model performance on validation and test sets
6. Compare CNN against MLP
7. Experiment with image augmentation and assess its impact on model performance

### Learning Focus

Understand how convolutional layers, feature maps, pooling, and fully connected layers work together to extract image features and perform classification. 

### Limitations

This is an educational project and is not intended for clinical diagnosis.

## 01. Dataset

NIH ChestX-ray14 subset of 3241 images, will be split following the MLP model for consistency and to allow for comparisons between the 2 models.

In [13]:
import torch
import pandas as pd

# Load preprocessed tensors
data = torch.load(
    "../../data/processed/mlp_images_64x64.pt",
    map_location="cpu",
    weights_only=True)

X = data["X"]
y = data["y"]
image_names = data["image_names"]
patient_ids = data["patient_ids"]  
splits = data["splits"]  

print("Images:", X.shape)
print("Labels:", y.shape)
print("Number of patients:", len(patient_ids))
print("Number of splits:", len(splits))
print("Data type:", X.dtype)
print("Minimum:", X.min().item())
print("Maximum:", X.max().item())

# Load dataset metadata
df = pd.read_csv("../../data/processed/mlp_dataset.csv")
print("Columns:", df.columns.tolist())
print(f"Split distribution: {df.groupby(['Split', 'Label']).size()}")


Images: torch.Size([3241, 1, 64, 64])
Labels: torch.Size([3241])
Number of patients: 3241
Number of splits: 3241
Data type: torch.float32
Minimum: 0.0
Maximum: 1.0
Columns: ['Image Index', 'Patient ID', 'Label', 'Split']
Split distribution: Split  Label
test   0         329
       1         150
train  0        1939
       1         269
val    0         486
       1          68
dtype: int64


## 02. Data Preparation

We will confirm that data for CNN model development is identical to the data used for MLP model development:
- 3241 images, with matching tensor and CSV 
- 2208 training, 554 validation, and 479 test images
- Labels encoded as 0 and 1
- Original patient-level splits preserved
- No need to preprocess or split data again

In [15]:
from collections import Counter

# Inspect split and label values
print("Split values:", Counter(splits))
print("Label values:", Counter(y.tolist()))

print("First 5 image names:", image_names[:5])
print("First 5 patient IDs:", patient_ids[:5])

# Verify CSV metadata
print("CSV columns:", df.columns.tolist())
print("Split and label distribution:")
print(df.groupby(["Split", "Label"]).size())
print("First 5 rows:")
display(df.head())

# Check tensor and CSV ordering match
print("Tensor images:", len(X))
print("CSV rows:", len(df))

print("Image names match:",
      image_names == df["Image Index"].tolist())

Split values: Counter({'train': 2208, 'val': 554, 'test': 479})
Label values: Counter({0: 2754, 1: 487})
First 5 image names: ['00000002_000.png', '00000005_000.png', '00000005_001.png', '00000005_002.png', '00000005_003.png']
First 5 patient IDs: [2, 5, 5, 5, 5]
CSV columns: ['Image Index', 'Patient ID', 'Label', 'Split']
Split and label distribution:
Split  Label
test   0         329
       1         150
train  0        1939
       1         269
val    0         486
       1          68
dtype: int64
First 5 rows:


,Image Index,Patient ID,Label,Split
0,00000002_000.png,2,0,train
1,00000005_000.png,5,0,train
2,00000005_001.png,5,0,train
3,00000005_002.png,5,0,train
4,00000005_003.png,5,0,train


Tensor images: 3241
CSV rows: 3241
Image names match: True


## 03. Model Architecture

### 3.1 Creating PyTorch Datasets and DataLoaders

Datasets store and provide access to individual sample and their labels.
DataLoader retrieves samples from the Dataset and groups them into batches. Training will be performed on these manageable subsets of data.

In [17]:
from torch.utils.data import TensorDataset, DataLoader, Subset

# Create a dataset containing images and labels
dataset = TensorDataset(X, y)

# Get indices for each split
train_idx = [i for i, s in enumerate(splits) if s == "train"]
val_idx = [i for i, s in enumerate(splits) if s == "val"]
test_idx = [i for i, s in enumerate(splits) if s == "test"]

# Create separate datasets
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Inspect 1 batch
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Labels in batch:", labels.unique())

Training batches: 69
Validation batches: 18
Test batches: 15
Image batch shape: torch.Size([32, 1, 64, 64])
Label batch shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64
Labels in batch: tensor([0, 1])


There are 69 training batches, 18 validation batches, and 15 test batches of 32 images each. 

Each image has 1 channel (grayscale), 64 pixels in height and 64 pixels in width (64x64), with the labels containing corresponding diagnosis class for each image.

### 3.2 Building the CNN Architecture

1. First convolution layer (nn.Conv2d)
2. ReLU activation to introduce non-linearity
3. MaxPooling (nn.MaxPool)
4. Second convolution layer (nn.Conv2d)
5. ReLU activation
6. MaxPooling
7. nn.Flatten
8. nn.Linear

In [34]:
import torch.nn as nn

# CNN Model
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(8192, 2)) #(32*16*16)

## 04. Training CNN model

### 4.1 Configure Loss Function, Optimizer, and Device

In [35]:
import torch.optim as optim

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

Using device: cpu


### 4.2 Training Loop for 10 epochs

In [36]:
import copy 

NUM_EPOCHS = 10
cnn_train_losses = []
cnn_val_losses = []
best_val_loss = float("inf")
best_model_state = None

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
    epoch_train_loss = running_train_loss / len(train_loader)
    cnn_train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
    epoch_val_loss = running_val_loss / len(val_loader)
    cnn_val_losses.append(epoch_val_loss)

    # Best model with least loss
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

    # Results
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )            
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")

# Restore best model
model.load_state_dict(best_model_state)

Epoch [1/10] Train Loss: 0.3771 Val Loss: 0.3964
Epoch [2/10] Train Loss: 0.3447 Val Loss: 0.3403
Epoch [3/10] Train Loss: 0.3358 Val Loss: 0.3323
Epoch [4/10] Train Loss: 0.3280 Val Loss: 0.3346
Epoch [5/10] Train Loss: 0.3171 Val Loss: 0.3261
Epoch [6/10] Train Loss: 0.3220 Val Loss: 0.3325
Epoch [7/10] Train Loss: 0.3144 Val Loss: 0.3322
Epoch [8/10] Train Loss: 0.3005 Val Loss: 0.3391
Epoch [9/10] Train Loss: 0.2983 Val Loss: 0.3328
Epoch [10/10] Train Loss: 0.2919 Val Loss: 0.3424
Best epoch: 5
Best validation loss: 0.3261


<All keys matched successfully>

Validation loss reaches its lowest point at epoch 5 (0.3261), then deteoriates while training loss decreases continually. This is an early sign of overfitting. 

Hence, we have saved and restored the best model (epoch 5), and will be using its weights. This model will be used for evaluation, rather than the final epoch 10 model. 

Next, we will evaluate this model on the validation dataset.

## 05. Evaluate CNN Model

Evaluate validation predictions

In [40]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

# Switch model to evaluation mode
model.eval()

cnn_predictions = []
cnn_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        # Select class with higher logit
        predictions = torch.argmax(outputs, dim=1)
        cnn_predictions.extend(predictions.cpu().numpy())
        cnn_labels.extend(labels.cpu().numpy())

# Confusion matrix
cm = confusion_matrix(cnn_labels, cnn_predictions)
print(cm)

# Evaluation metrics
accuracy = accuracy_score(cnn_labels, cnn_predictions)
classification_metrics = classification_report(cnn_labels, cnn_predictions, target_names=["No Finding", "Effusion"], zero_division=0)
precision = precision_score(cnn_labels, cnn_predictions, zero_division=0)
recall = recall_score(cnn_labels, cnn_predictions, zero_division=0)
f1 = f1_score(cnn_labels, cnn_predictions, zero_division=0)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 score: {f1:.4f}")
print("Classification Report:", classification_metrics)

[[471  15]
 [ 59   9]]
Accuracy: 0.8664
Precision: 0.3750
Recall: 0.1324
F1 score: 0.1957
Classification Report:               precision    recall  f1-score   support

  No Finding       0.89      0.97      0.93       486
    Effusion       0.38      0.13      0.20        68

    accuracy                           0.87       554
   macro avg       0.63      0.55      0.56       554
weighted avg       0.83      0.87      0.84       554



Our current CNN model (epoch 5) predicted 471 TNs, 59 FNs, 15 FPs, and 9 TPs. It has detected only 9 out of 68 actual Effusion images, missing the remaining 59. It correctly identified 471 of 486 No Finding images. 

This model is good at recognising "No Finding", but struggles with the "Effusion" minority class. 

| Metrics | MLP | CNN |
| ------- | --- | --- |
|Accuracy | 0.8755 | 0.8664 |
|Precision | 0.00 | 0.3750 |
|Recall | 0.00 | 0.1324 |
|F1 | 0.00 | 0.1957 |
Compared to the MLP model, the CNN model shows improvement in the precision, recall, and F1 score, suggesting that it has learned more useful image features through convolutional filters. Although its accuracy is slightly lower, we should remember that the MLP model predicted all images as "No Finding". Its accuracy was therefore driven by the majority class, rather than its ability to detect Effusion. 

Next, we will attempt to handle class imbalance using class weighting to see whether it improves the CNN model's ability to detect Effusion.

## 06. Experiments: Handling Class Imbalance

The training dataset contains 1939 "No Finding" and 269 "Effusion" images. That's a 7.2:1 class imbalance. 

### 6.1 Introduce Class Weights

With class weighting, the loss function will penalise mistakes on Effusion more heavily. 

In [41]:
class_counts = torch.bincount(y[train_idx])
class_weights = len(train_idx) / (2 * class_counts.float())
print("Class counts:", class_counts)
print("Class weights:", class_weights)
# Effusion receives much higher weight due to it being minority class

Class counts: tensor([1939,  269])
Class weights: tensor([0.5694, 4.1041])


### 6.2 Train Weighted CNN

In [42]:
# Start with fresh CNN model so it doesn't retain what it learned during baseline training
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(8192, 2)) #(32*16*16)
model = model.to(device)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Update loss function to penalise incorrect effusion predictions more heavily
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

In [43]:
# Retrain new weighted CNN model using same 10-epoch training and validation loop
NUM_EPOCHS = 10
weighted_cnn_train_losses = []
weighted_cnn_val_losses = []
weighted_best_val_loss = float("inf")
weighted_best_model_state = None

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    weighted_running_train_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        weighted_running_train_loss += loss.item()
    weighted_epoch_train_loss = weighted_running_train_loss / len(train_loader)
    weighted_cnn_train_losses.append(weighted_epoch_train_loss)

    # Validation
    model.eval()
    weighted_running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            weighted_running_val_loss += loss.item()
    weighted_epoch_val_loss = weighted_running_val_loss / len(val_loader)
    weighted_cnn_val_losses.append(weighted_epoch_val_loss)

    # Best model with least loss
    if weighted_epoch_val_loss < weighted_best_val_loss:
        weighted_best_val_loss = weighted_epoch_val_loss
        weighted_best_model_state = copy.deepcopy(model.state_dict())
        weighted_best_epoch = epoch + 1

    # Results
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {weighted_epoch_train_loss:.4f} "
        f"Val Loss: {weighted_epoch_val_loss:.4f}"
    )            
print(f"Best epoch: {weighted_best_epoch}")
print(f"Best validation loss: {weighted_best_val_loss:.4f}")

# Restore best model
model.load_state_dict(weighted_best_model_state)

Epoch [1/10] Train Loss: 0.6753 Val Loss: 0.5886
Epoch [2/10] Train Loss: 0.6179 Val Loss: 0.5538
Epoch [3/10] Train Loss: 0.5903 Val Loss: 0.5256
Epoch [4/10] Train Loss: 0.5681 Val Loss: 0.5400
Epoch [5/10] Train Loss: 0.5501 Val Loss: 0.5238
Epoch [6/10] Train Loss: 0.5432 Val Loss: 0.5121
Epoch [7/10] Train Loss: 0.5292 Val Loss: 0.4985
Epoch [8/10] Train Loss: 0.5064 Val Loss: 0.5169
Epoch [9/10] Train Loss: 0.4950 Val Loss: 0.5491
Epoch [10/10] Train Loss: 0.4886 Val Loss: 0.5143
Best epoch: 7
Best validation loss: 0.4985


<All keys matched successfully>

Validation loss reaches its lowest point at epoch 7 (0.4985), then fluctuated, while training loss continued decreasing. This may suggest some overfitting towards the later epochs. (Note: weighted loss (0.4985) and unweighted loss (0.3261) are not directly comparable because class weights changes the loss calculation itself.) 

Hence, we have saved and restored the best model (epoch 7), and will be using its weights. This model will be used for evaluation, rather than the final epoch 10 model. 

Next, we will evaluate this model on the validation dataset.

### 6.3 Evaluate Weighted CNN

In [46]:
# Switch model to evaluation mode
model.eval()

weighted_cnn_predictions = []
weighted_cnn_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        # Select class with higher logit
        predictions = torch.argmax(outputs, dim=1)
        weighted_cnn_predictions.extend(predictions.cpu().numpy())
        weighted_cnn_labels.extend(labels.cpu().numpy())

# Confusion matrix
cm = confusion_matrix(weighted_cnn_labels, weighted_cnn_predictions)
print(cm)

# Evaluation metrics
accuracy = accuracy_score(weighted_cnn_labels, weighted_cnn_predictions)
precision = precision_score(weighted_cnn_labels, weighted_cnn_predictions, zero_division=0)
recall = recall_score(weighted_cnn_labels, weighted_cnn_predictions, zero_division=0)
f1 = f1_score(weighted_cnn_labels, weighted_cnn_predictions, zero_division=0)
classification_metrics = classification_report(weighted_cnn_labels, weighted_cnn_predictions, target_names=["No Finding", "Effusion"], zero_division=0)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 score: {f1:.4f}")
print("Classification Report:", classification_metrics)

[[402  84]
 [ 20  48]]
Accuracy: 0.8123
Precision: 0.3636
Recall: 0.7059
F1 score: 0.4800
Classification Report:               precision    recall  f1-score   support

  No Finding       0.95      0.83      0.89       486
    Effusion       0.36      0.71      0.48        68

    accuracy                           0.81       554
   macro avg       0.66      0.77      0.68       554
weighted avg       0.88      0.81      0.84       554



|Validation metric	|Weighted MLP | Baseline CNN|Weighted CNN|
| ------ | -------| ------| ------- |
|Accuracy |77.26%| 86.64% |  81.23% |
|Precision |32.53%| 37.50% | 36.36% |
|Recall |79.41%| 13.24% | 70.59% |
|F1	| 46.15%| 19.57% | 48.00% |

The weighted CNN model (epoch 7) correctly identified 48 out of 68 actual Effusion images (TP), only missing 20 (FN), as compared with 59 in the baseline CNN model. However, it incorrectly classified 84 No Finding images as Effusion (FP). 

Applying class weighted increased Effusion recall substantially, from 13.24% to 70.59%, while precision decreased only slightly, from 37.50% to 36.36%. This demonstrates the precision-recall trade-off: the model became more sensitive to Effusion, but generated more false positives.

The baseline CNN model was biased towards predicting the majority class. By assigning a heavier penalty to incorrect Effusion predictions, class weighting encouraged the model to pay more attention to the minority class. The CNN architecture remained unchanged, so this improvement was not due to learning new image features, but rather to optimising the same architecture under a different loss function.

When comparing the weighted MLP and weighted CNN, the MLP achieved higher Effusion recall (79.41% vs 70.59%), whereas the CNN achieved higher precision (36.36% vs 32.53%) and F1-score (48.00% vs 46.15%). This suggests that the CNN model offers a slightly better balance between precision and recall, although the difference in F1-score is small. Neither model clearly dominates across all metrics.